In [1]:
# ============================================================
# CELL 1 - Load Data & Prepare Features
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

DATA_PATH = "../data/processed/modeling_data.csv"
df = pd.read_csv(DATA_PATH)

target = "co2"
features = [
    "year", "population_co2", "gdp_co2", "primary_energy_consumption_energy",
    "fossil_share_energy", "renewables_share_energy", "renewables_electricity",
    "renewables_share_elec", "solar_share_energy", "wind_share_energy",
    "hydro_share_energy", "gdp_per_capita", "energy_per_capita"
]

X = df[features]
y = df[target]

print("=" * 60)
print("REGRESSION MODELING")
print("=" * 60)
print("Features:", X.shape)
print("Target:", y.shape)
print("Total samples:", len(df))

REGRESSION MODELING
Features: (1817, 13)
Target: (1817,)
Total samples: 1817


In [2]:
# ============================================================
# CELL 2 - 70/15/15 Data Split
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("=" * 60)
print("DATA SPLIT")
print("=" * 60)
print("Training:", X_train.shape, f"({len(X_train)/len(X)*100:.1f}%)")
print("Validation:", X_val.shape, f"({len(X_val)/len(X)*100:.1f}%)")
print("Testing:", X_test.shape, f"({len(X_test)/len(X)*100:.1f}%)")

DATA SPLIT
Training: (1271, 13) (70.0%)
Validation: (273, 13) (15.0%)
Testing: (273, 13) (15.0%)


In [3]:
# ============================================================
# CELL 3 - Train Regression Models
# ============================================================

print("=" * 60)
print("MODEL TRAINING")
print("=" * 60)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
}

predictions = {}

for name, model in models.items():
    if name == "Linear Regression":
        model.fit(X_train_scaled, y_train)
        predictions[name] = model.predict(X_val_scaled)
    else:
        model.fit(X_train, y_train)
        predictions[name] = model.predict(X_val)

    print(f"{name}: trained successfully")

MODEL TRAINING
Linear Regression: trained successfully
Random Forest: trained successfully
Gradient Boosting: trained successfully


In [4]:
# ============================================================
# CELL 4 - Validation Performance
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=" * 60)
print("VALIDATION PERFORMANCE")
print("=" * 60)

validation_results = []

for name, pred in predictions.items():
    mae = mean_absolute_error(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2 = r2_score(y_val, pred)
    validation_results.append([name, mae, rmse, r2])

validation_results = pd.DataFrame(validation_results, columns=["Model", "MAE", "RMSE", "R²"])
display(validation_results.sort_values("RMSE"))

VALIDATION PERFORMANCE


,Model,MAE,RMSE,R²
2,Gradient Boosting,20.937090,41.623754,0.998726
1,Random Forest,16.312055,60.711194,0.997289
0,Linear Regression,61.561040,101.114158,0.992480


In [5]:
# ============================================================
# CELL 5 - Select Best Model & Test Evaluation
# ============================================================

best_model_name = validation_results.loc[validation_results["RMSE"].idxmin(), "Model"]
best_model = models[best_model_name]

if best_model_name == "Linear Regression":
    test_pred = best_model.predict(X_test_scaled)
else:
    test_pred = best_model.predict(X_test)

test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
test_r2 = r2_score(y_test, test_pred)

print("=" * 60)
print("FINAL TEST EVALUATION")
print("=" * 60)
print("Best model:", best_model_name)
print("Test MAE:", round(test_mae, 4))
print("Test RMSE:", round(test_rmse, 4))
print("Test R²:", round(test_r2, 4))

FINAL TEST EVALUATION
Best model: Gradient Boosting
Test MAE: 22.5743
Test RMSE: 68.9022
Test R²: 0.9973
